In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, IntegerType

# 屏蔽警告
import warnings
import sys
warnings.filterwarnings('ignore')
sys.dont_write_bytecode = True
if not sys.warnoptions:
    warnings.simplefilter("ignore")

spark = SparkSession.builder \
    .appName("OA_Workflow_Efficiency_Analysis") \
    .enableHiveSupport() \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.hive.convertMetastoreParquet", "true") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

# 表名定义
TABLE_REQUEST_BASE = "ods.ods_oa_workflow_requestbase_i_1h"
TABLE_CURRENT_OPERATOR = "ods.ods_oa_workflow_currentoperator_i_1h"
TABLE_WORKFLOW_BASE = "ods.ods_oa_workflow_base_f_1d"
TABLE_REQUEST_LOG = "ods.ods_oa_workflow_requestlog_i_1h"
TABLE_HRM_RESOURCE = "ods.ods_oa_hrmresource_f_1d"
TABLE_HRM_DEPARTMENT = "ods.ods_oa_hrmdepartment_f_1d"
TABLE_HRM_JOBTITLES = "ods.ods_oa_hrmjobtitles_f_1d"
TABLE_HRM_SUBCOMPANY = "ods.ods_oa_hrmsubcompany_f_1d"
TABLE_NODE_BASE = "ods.ods_oa_workflow_nodebase_f_1d"
TABLE_WORKFLOW_FLOWNODE = "ods.ods_oa_workflow_flownode_f_1d"
target_table = "dwd.dwd_oa_workflow_efficiency_result_f_1h"

# 加载原始表
df_request_base = spark.table(TABLE_REQUEST_BASE).alias("T1")
df_current_operator = spark.table(TABLE_CURRENT_OPERATOR).alias("T2")
df_workflow_base = spark.table(TABLE_WORKFLOW_BASE).alias("T3")
df_request_log = spark.table(TABLE_REQUEST_LOG).alias("T7")
df_hrm_resource = spark.table(TABLE_HRM_RESOURCE).alias("T4")
df_hrm_department = spark.table(TABLE_HRM_DEPARTMENT).alias("T6")
df_hrm_jobtitles = spark.table(TABLE_HRM_JOBTITLES).alias("T5")
df_hrm_subcompany = spark.table(TABLE_HRM_SUBCOMPANY).alias("Company")
df_nodebase = spark.table(TABLE_NODE_BASE).alias("NB")
df_workflow_flownode = spark.table(TABLE_WORKFLOW_FLOWNODE).alias("FN")

# ==================== 硬编码2024年节假日及调休周末列表 ====================
# 法定节假日（放假日期，包含调休连休的周末）
holiday_list_2024 = [
    # 2022
    '2022-01-01', '2022-01-02', '2022-01-03',
    '2022-01-31', '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-04', '2022-02-05', '2022-02-06',
    '2022-04-03', '2022-04-04', '2022-04-05',
    '2022-04-30', '2022-05-01', '2022-05-02', '2022-05-03', '2022-05-04',
    '2022-06-03', '2022-06-04', '2022-06-05',
    '2022-09-10', '2022-09-11', '2022-09-12',
    '2022-10-01', '2022-10-02', '2022-10-03', '2022-10-04', '2022-10-05', '2022-10-06', '2022-10-07',
    # 2023
    '2023-01-01', '2023-01-02',
    '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24', '2023-01-25', '2023-01-26', '2023-01-27',
    '2023-04-05',
    '2023-04-29', '2023-04-30', '2023-05-01', '2023-05-02', '2023-05-03',
    '2023-06-22', '2023-06-23', '2023-06-24',
    '2023-09-29', '2023-09-30', '2023-10-01', '2023-10-02', '2023-10-03', '2023-10-04', '2023-10-05', '2023-10-06',
    # 2024
    '2024-01-01',
    '2024-02-10', '2024-02-11', '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15', '2024-02-16',
    '2024-04-04', '2024-04-05', '2024-04-06',
    '2024-05-01', '2024-05-02', '2024-05-03',
    '2024-06-10',
    '2024-09-15', '2024-09-16', '2024-09-17',
    '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04', '2024-10-05', '2024-10-06', '2024-10-07',
    # 2025
    '2025-01-01',
    '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-03',
    '2025-04-04', '2025-04-05', '2025-04-06',
    '2025-05-01', '2025-05-02', '2025-05-03',
    '2025-05-31', '2025-06-01', '2025-06-02',
    '2025-10-01', '2025-10-02', '2025-10-03', '2025-10-04', '2025-10-05', '2025-10-06', '2025-10-07',
    # 2026 (预测)
    '2026-01-01',
    '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18', '2026-02-19', '2026-02-20', '2026-02-21',
    '2026-04-04', '2026-04-05', '2026-04-06',
    '2026-05-01', '2026-05-02', '2026-05-03',
    '2026-06-20', '2026-06-21', '2026-06-22',
    '2026-09-27', '2026-09-28', '2026-09-29',
    '2026-10-01', '2026-10-02', '2026-10-03', '2026-10-04', '2026-10-05', '2026-10-06', '2026-10-07',
]

# 调休上班的周末（原本是周末，但需要上班）
work_weekend_list_2024 = [
    # 2022
    '2022-01-29', '2022-01-30',
    '2022-04-02',
    '2022-04-24', '2022-05-07',
    '2022-10-08', '2022-10-09',
    # 2023
    '2023-01-28', '2023-01-29',
    '2023-04-23',
    '2023-05-06',
    '2023-06-25',
    '2023-10-07', '2023-10-08',
    # 2024
    '2024-02-04', '2024-02-18',
    '2024-04-07',
    '2024-04-28', '2024-05-11',
    '2024-09-14',
    '2024-09-29', '2024-10-12',
    # 2025
    '2025-01-26',
    '2025-02-08',
    '2025-04-27',
    '2025-05-06',
    '2025-06-28', '2025-10-11',
    # 2026 (预测)
    '2026-02-14', '2026-02-22',
    '2026-04-03',
    '2026-04-26', '2026-05-09',
    '2026-06-13',
    '2026-09-20', '2026-10-10',
]

# ==================== 1. 计算全流程起止时间（无筛选） ====================
df_process_time_raw = df_current_operator.select(
    F.col("requestid"),
    F.to_timestamp(F.concat_ws(" ", F.col("receivedate"), F.col("receivetime")), "yyyy-MM-dd HH:mm:ss").alias("receive_time"),
    F.to_timestamp(F.concat_ws(" ", F.col("operatedate"), F.col("operatetime")), "yyyy-MM-dd HH:mm:ss").alias("operate_time")
).filter(F.col("receive_time").isNotNull() & F.col("operate_time").isNotNull())

df_process_time = df_process_time_raw.groupBy("requestid").agg(
    F.min("receive_time").alias("process_start_time"),
    F.max("operate_time").alias("process_end_time")
)

# ==================== 2. 审批节点基础数据（筛选非填报人/知会/办结） ====================
df_node_base = df_current_operator.alias("T2")\
    .join(df_nodebase.alias("NB"), F.col("T2.nodeid") == F.col("NB.id"), "left") \
    .filter(~F.lower(F.coalesce(F.col("NB.nodename"), F.lit(""))).rlike(r'填报人|知会|办结')) \
    .select(
        F.col("T2.requestid"),
        F.col("T2.nodeid").alias("node_id"),
        F.col("T2.userid"),
        F.col("NB.nodename").alias("nodename"),
        F.to_timestamp(F.concat_ws(" ", F.col("T2.receivedate"), F.col("T2.receivetime")), "yyyy-MM-dd HH:mm:ss").alias("receive_time"),
        F.to_timestamp(F.concat_ws(" ", F.col("T2.operatedate"), F.col("T2.operatetime")), "yyyy-MM-dd HH:mm:ss").alias("operate_time")
    ).filter(F.col("receive_time").isNotNull() & F.col("operate_time").isNotNull() & (F.col("operate_time") >= F.col("receive_time")))

# 2.1 审批开始时间（第一个审批节点接收时间）
df_approval_start = df_node_base.groupBy("requestid").agg(
    F.min("receive_time").alias("StartApprovalTime"),
    F.max("operate_time").alias("max_operate_time")   # 后备结束时间（操作时间）
)

# 2.2 从请求日志中获取结束时间（logtype in ('0','2') 且节点不是结束节点、不是填报人/知会/办结）
df_end_approval_log = df_request_log.alias("LOG") \
    .join(df_nodebase.alias("NB"), F.col("LOG.nodeid") == F.col("NB.id"), "left") \
    .filter(
        F.col("LOG.logtype").isin(['0', '2']) &
        ((F.col("NB.isend") != 1) | F.col("NB.isend").isNull()) &
        (~F.lower(F.coalesce(F.col("NB.nodename"), F.lit(""))).rlike(r'填报人|知会|办结'))
    ) \
    .select(
        F.col("LOG.requestid"),
        F.to_timestamp(F.concat_ws(" ", F.col("LOG.operatedate"), F.col("LOG.operatetime")), "yyyy-MM-dd HH:mm:ss").alias("operate_time")
    ) \
    .filter(F.col("operate_time").isNotNull()) \
    .groupBy("requestid") \
    .agg(F.max("operate_time").alias("EndApprovalTime"))

# 2.3 合并：优先取日志表中的最大操作时间，否则用审批节点的最大操作时间（后备）
df_approval_time = df_approval_start \
    .join(df_end_approval_log, on="requestid", how="left") \
    .withColumn("EndApprovalTime", F.coalesce(F.col("EndApprovalTime"), F.col("max_operate_time"))) \
    .select("requestid", "StartApprovalTime", "EndApprovalTime")

# ==================== 3. 岗位时长计算（剔除周末和节假日，考虑调休） ====================
# 关联岗位信息
df_node_job = df_node_base.join(df_hrm_resource.alias("T4"), F.col("userid") == F.col("T4.id"), "left") \
    .join(df_hrm_jobtitles.alias("T5"), F.col("T4.jobtitle") == F.col("T5.id"), "left") \
    .select(
        F.col("requestid"),
        F.col("node_id"),
        F.col("receive_time"),
        F.col("operate_time"),
        F.col("T4.jobactivitydesc").alias("job_level"),
        F.col("T5.jobtitlemark").alias("job_title_name"),
        F.expr("sequence(to_date(receive_time), to_date(operate_time), interval 1 day)").alias("date_array")
    ).filter((F.col("job_level").isNotNull()) | (F.col("job_title_name").isNotNull()))

# 拆分每日
df_node_daily = df_node_job.withColumn("dt", F.explode("date_array"))

# 判断是否为工作日：在调休周末列表中 或 （不在节假日列表中 且 不是周末）
df_node_daily = df_node_daily.withColumn(
    "is_workday",
    (F.col("dt").isin(work_weekend_list_2024)) | 
    (~F.col("dt").isin(holiday_list_2024) & ~F.dayofweek(F.col("dt")).isin(1,7))
)

# 计算每天的工作秒数（若为工作日则计算实际耗时，否则0）
df_node_daily_calc = df_node_daily \
    .withColumn("dt_start_unix", F.unix_timestamp(F.col("dt"))) \
    .withColumn("dt_end_unix", F.unix_timestamp(F.date_add(F.col("dt"), 1)) - 1) \
    .withColumn("receive_unix", F.unix_timestamp(F.col("receive_time"))) \
    .withColumn("operate_unix", F.unix_timestamp(F.col("operate_time"))) \
    .withColumn(
        "daily_seconds",
        F.when(F.col("is_workday"),
               F.greatest(F.least(F.col("operate_unix"), F.col("dt_end_unix")),
                         F.greatest(F.col("receive_unix"), F.col("dt_start_unix"))) -
               F.greatest(F.col("receive_unix"), F.col("dt_start_unix"))
              ).otherwise(F.lit(0))
    )

# ===== 新增自然日秒数计算 =====
df_node_daily_calc = df_node_daily_calc.withColumn(
    "natural_seconds",
    F.when((F.col("dt") == F.to_date(F.col("receive_time"))) & (F.col("dt") == F.to_date(F.col("operate_time"))),
           F.col("operate_unix") - F.col("receive_unix"))
    .when(F.col("dt") == F.to_date(F.col("receive_time")),
          F.unix_timestamp(F.date_add(F.col("dt"), 1)) - 1 - F.col("receive_unix"))
    .when(F.col("dt") == F.to_date(F.col("operate_time")),
          F.col("operate_unix") - F.unix_timestamp(F.col("dt")))
    .otherwise(F.lit(86400))
)

# 岗位工作日时长统计函数
def calc_job_level_hours(df, regex, alias):
    df_filter = df.filter(F.col("job_level").rlike(regex))
    df_stats = df_filter.groupBy("requestid").agg(
        F.sum("daily_seconds").alias(f"{alias}_total"),
        F.countDistinct("node_id").alias(f"{alias}_count")
    ).fillna(0).withColumn(
        f"avg_{alias}_workday_hours",
        (F.col(f"{alias}_total") / F.col(f"{alias}_count") / 3600).cast(DoubleType())
    ).select("requestid", f"avg_{alias}_workday_hours")
    return df_stats

# ===== 新增自然日时长统计函数 =====
def calc_job_level_natural_hours(df, regex, alias):
    df_filter = df.filter(F.col("job_level").rlike(regex))
    df_stats = df_filter.groupBy("requestid").agg(
        F.sum("natural_seconds").alias(f"{alias}_natural_total"),
        F.countDistinct("node_id").alias(f"{alias}_count")
    ).fillna(0).withColumn(
        f"avg_{alias}_natural_hours",
        (F.col(f"{alias}_natural_total") / F.col(f"{alias}_count") / 3600).cast(DoubleType())
    ).select("requestid", f"avg_{alias}_natural_hours")
    return df_stats

# 计算各岗位工作日平均时长
df_op_stats = calc_job_level_hours(df_node_daily_calc, "^P|^O", "op")
df_m1_stats = calc_job_level_hours(df_node_daily_calc, "^M1$", "m1")
df_m2_stats = calc_job_level_hours(df_node_daily_calc, "^M2$", "m2")
df_m3m4_stats = calc_job_level_hours(df_node_daily_calc, "^M3|^M4", "m3m4")
df_m5m6_stats = calc_job_level_hours(df_node_daily_calc, "^M5|^M6", "m5m6")
df_president_stats = calc_job_level_hours(df_node_daily_calc, "^总裁$", "president")
df_ceo_stats = calc_job_level_hours(df_node_daily_calc, "^CEO$", "ceo")
df_vp_cfo_stats = calc_job_level_hours(df_node_daily_calc, "^M8$", "vp_cfo")

# ===== 计算各岗位自然日平均时长 =====
df_op_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^P|^O", "op")
df_m1_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^M1$", "m1")
df_m2_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^M2$", "m2")
df_m3m4_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^M3|^M4", "m3m4")
df_m5m6_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^M5|^M6", "m5m6")
df_president_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^总裁$", "president")
df_ceo_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^CEO$", "ceo")
df_vp_cfo_natural_stats = calc_job_level_natural_hours(df_node_daily_calc, "^M8$", "vp_cfo")

# ==================== 4. 其他维度子表（均按 requestid 聚合为唯一行） ====================
# 4.1 提单人部门全路径
df_dept_org = df_hrm_department.alias("Dep") \
    .join(df_hrm_department.alias("supDept"), F.col("Dep.supdepid") == F.col("supDept.id"), "left") \
    .join(df_hrm_department.alias("supDept1"), F.col("supDept.supdepid") == F.col("supDept1.id"), "left") \
    .join(df_hrm_department.alias("supDept2"), F.col("supDept1.supdepid") == F.col("supDept2.id"), "left") \
    .join(df_hrm_department.alias("supDept3"), F.col("supDept2.supdepid") == F.col("supDept3.id"), "left") \
    .join(df_hrm_subcompany.alias("Company"), F.col("Dep.subcompanyid1") == F.col("Company.id"), "left") \
    .filter((F.col("Dep.canceled").isNull() | (F.col("Dep.canceled") == 0)) &
            (F.col("Company.id").isNull() | ~F.col("Company.id").isin(11, 20, 21, 22, 18))) \
    .select(
        F.col("Dep.id").alias("dept_id"),
        F.concat(
            F.coalesce(F.col("Company.subcompanyname"), F.lit("")), F.lit("||"),
            F.coalesce(F.concat(F.col("supDept3.departmentname"), F.lit(">")), F.lit("")),
            F.coalesce(F.concat(F.col("supDept2.departmentname"), F.lit(">")), F.lit("")),
            F.coalesce(F.concat(F.col("supDept1.departmentname"), F.lit(">")), F.lit("")),
            F.coalesce(F.concat(F.col("supDept.departmentname"), F.lit(">")), F.lit("")),
            F.coalesce(F.col("Dep.departmentname"), F.lit(""))
        ).alias("FullDeptPath")
    ).withColumn(
        "DeptFullPath",
        F.when(F.col("FullDeptPath").endswith(">"),
               F.expr("substring(FullDeptPath, 1, length(FullDeptPath) - 1)")
              ).otherwise(F.col("FullDeptPath"))
    )

# 提单人部门（每个 requestid 唯一，取第一个提单人部门）
df_bill_dept = df_current_operator.filter(F.col("showorder") == -1) \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .select(F.col("requestid"), F.col("departmentid").alias("dept_id")) \
    .dropDuplicates(["requestid"])

df_org_data = df_bill_dept.join(df_dept_org, on="dept_id", how="left") \
    .select("requestid", "DeptFullPath") \
    .dropDuplicates(["requestid"])

# 4.2 流程名称（唯一）
df_wf_name = df_current_operator.join(df_workflow_base, F.col("workflowid") == F.col("T3.id"), "left") \
    .groupBy("requestid").agg(F.first("workflowname", ignorenulls=True).alias("WorkFlowName"))

# 4.3 是否退回（唯一）
df_return_data = df_request_log.filter(F.col("logtype") == "3") \
    .groupBy("requestid").agg(
        F.when(F.count("*") >= 1, F.lit("是")).otherwise(F.lit("否")).alias("IsReturn")
    )

# 4.4 退回人员姓名（唯一）
df_return_person = df_request_log.filter(F.col("logtype") == "3") \
    .join(df_hrm_resource, F.col("operator") == F.col("id"), "left") \
    .filter(F.col("lastname").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws(",", F.collect_set("lastname")).alias("return_person_names")
    )

# 4.5 审批人员姓名（排除提单人，唯一）
df_submit_person = df_current_operator.filter(F.col("showorder") == -1) \
    .select("requestid", F.col("userid").alias("submit_userid")) \
    .dropDuplicates(["requestid"])

df_approve_person = df_request_log.filter(F.col("logtype") != "3") \
    .join(df_submit_person, on="requestid", how="left") \
    .filter(F.col("operator").isNotNull() &
            (F.col("operator") != F.coalesce(F.col("submit_userid"), F.lit("")))) \
    .join(df_hrm_resource, F.col("operator") == F.col("id"), "left") \
    .filter(F.col("lastname").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws(",", F.collect_set("lastname")).alias("approve_person_names")
    )

# 4.6 工作流固定审批节点数量（唯一）
df_fixed_node_count = df_workflow_base.alias("T3") \
    .join(df_workflow_flownode.alias("FN"), F.col("T3.id") == F.col("FN.workflowid"), "left") \
    .groupBy("T3.id").agg(
        (F.coalesce(F.count("FN.workflowid"), F.lit(0)) - 2).alias("fixed_approval_node_count")
    ).withColumnRenamed("id", "workflow_id_for_join")

# 4.7 业务审批部门（提单人部门，唯一）
df_business_dept = df_current_operator.filter(F.col("showorder") == -1) \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .join(df_hrm_department, F.col("departmentid") == F.col("T6.id"), "left") \
    .groupBy("requestid").agg(F.first("departmentname").alias("business_approve_dept"))

# 4.8 职能审批部门（所有审批节点部门，唯一）
df_func_dept = df_current_operator.filter(F.col("showorder") != -1) \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .join(df_hrm_department, F.col("departmentid") == F.col("T6.id"), "left") \
    .filter(F.col("departmentname").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws("、", F.collect_set("departmentname")).alias("func_approve_dept")
    )

# 4.9 所有审批部门（包含提单人）
df_all_dept = df_current_operator \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .join(df_hrm_department, F.col("departmentid") == F.col("T6.id"), "left") \
    .filter(F.col("departmentname").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws("、", F.collect_set("departmentname")).alias("all_approve_dept")
    )

# 4.10 审批角色（唯一）
df_approve_role = df_current_operator.filter(F.col("showorder") != -1) \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .join(df_hrm_jobtitles, F.col("jobtitle") == F.col("T5.id"), "left") \
    .filter(F.col("jobtitlemark").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws("、", F.collect_set("jobtitlemark")).alias("approve_role")
    )

# 4.11 审批统计（实际审批人数、节点数，唯一）
df_approve_stats = df_request_log \
    .filter(F.col("logtype").isin(['0', '2'])) \
    .groupBy("requestid") \
    .agg(
        F.count("operator").alias("actual_approve_person_times"),
        F.countDistinct("operator").alias("actual_approve_node_num")
    ) \
    .fillna(0)
# 4.12 岗位级别（唯一）
df_job_level = df_node_base \
    .join(df_hrm_resource, F.col("userid") == F.col("T4.id"), "left") \
    .filter(F.col("jobactivitydesc").isNotNull()) \
    .groupBy("requestid").agg(
        F.concat_ws("、", F.collect_set("jobactivitydesc")).alias("job_level")
    )

# ==================== 5. 构建主表（确保每个 requestid 唯一） ====================
time_str = '2022-01-01'
df_main = df_request_base.filter(
    (F.col("createdate") > time_str) & (
        (F.col("status") == "办结") | 
        (F.col("requestname").contains("预算"))
    )
).select("requestid", "requestname", "workflowid", "status", "requestmark")

# 依次左连接各维度子表
df_main = df_main.join(df_approval_time, on="requestid", how="left") \
    .join(df_process_time, on="requestid", how="left") \
    .join(df_return_data, on="requestid", how="left") \
    .join(df_return_person, on="requestid", how="left") \
    .join(df_approve_person, on="requestid", how="left") \
    .join(df_org_data, on="requestid", how="left") \
    .join(df_business_dept, on="requestid", how="left") \
    .join(df_func_dept, on="requestid", how="left") \
    .join(df_approve_role, on="requestid", how="left") \
    .join(df_approve_stats, on="requestid", how="left") \
    .join(df_wf_name, on="requestid", how="left") \
    .join(df_job_level, on="requestid", how="left") \
    .join(df_op_stats, on="requestid", how="left") \
    .join(df_m1_stats, on="requestid", how="left") \
    .join(df_m2_stats, on="requestid", how="left") \
    .join(df_m3m4_stats, on="requestid", how="left") \
    .join(df_m5m6_stats, on="requestid", how="left") \
    .join(df_president_stats, on="requestid", how="left") \
    .join(df_ceo_stats, on="requestid", how="left") \
    .join(df_vp_cfo_stats, on="requestid", how="left") \
    .join(df_op_natural_stats, on="requestid", how="left") \
    .join(df_m1_natural_stats, on="requestid", how="left") \
    .join(df_m2_natural_stats, on="requestid", how="left") \
    .join(df_m3m4_natural_stats, on="requestid", how="left") \
    .join(df_m5m6_natural_stats, on="requestid", how="left") \
    .join(df_president_natural_stats, on="requestid", how="left") \
    .join(df_ceo_natural_stats, on="requestid", how="left") \
    .join(df_vp_cfo_natural_stats, on="requestid", how="left") \
    .join(df_all_dept, on="requestid", how="left") \
    .join(df_fixed_node_count.alias("fnc"), F.col("workflowid") == F.col("fnc.workflow_id_for_join"), "left")

df_main = df_main.dropDuplicates(["requestid"])

# ==================== 6. 计算总时长（自然小时 & 工作小时，剔除周末和节假日，考虑调休） ====================
df_calc = df_main \
    .withColumn("start_unix", F.unix_timestamp(F.col("StartApprovalTime"))) \
    .withColumn("end_unix", F.unix_timestamp(F.col("EndApprovalTime"))) \
    .withColumn("start_date", F.to_date(F.col("StartApprovalTime"))) \
    .withColumn("end_date", F.to_date(F.col("EndApprovalTime"))) \
    .withColumn("date_array", F.expr("sequence(start_date, end_date, interval 1 day)")) \
    .withColumn("dt", F.explode("date_array")) \
    .withColumn(
        "is_workday",
        (F.col("dt").isin(work_weekend_list_2024)) | 
        (~F.col("dt").isin(holiday_list_2024) & ~F.dayofweek(F.col("dt")).isin(1,7))
    ) \
    .withColumn(
        "day_seconds",
        F.when(~F.col("is_workday"), F.lit(0))  # 非工作日全天为0
        .when((F.col("dt") == F.col("start_date")) & (F.col("dt") == F.col("end_date")),
              F.col("end_unix") - F.col("start_unix"))
        .when(F.col("dt") == F.col("start_date"),
              F.unix_timestamp(F.date_add(F.col("dt"), 1)) - 1 - F.col("start_unix"))
        .when(F.col("dt") == F.col("end_date"),
              F.col("end_unix") - F.unix_timestamp(F.col("dt")))
        .otherwise(F.lit(86400))  # 整天秒数
    )

# 聚合得到总工作秒数，同时保留其他字段
df_agg = df_calc.groupBy("requestid").agg(
    F.sum("day_seconds").alias("total_work_seconds"),
    F.first("StartApprovalTime").alias("StartApprovalTime"),
    F.first("EndApprovalTime").alias("EndApprovalTime"),
    F.first("process_start_time").alias("process_start_time"),
    F.first("process_end_time").alias("process_end_time"),
    F.first("requestname").alias("requestname"),
    F.first("WorkFlowName").alias("WorkFlowName"),
    F.first("IsReturn").alias("IsReturn"),
    F.first("DeptFullPath").alias("DeptFullPath"),
    F.first("business_approve_dept").alias("business_approve_dept"),
    F.first("func_approve_dept").alias("func_approve_dept"),
    F.first("all_approve_dept").alias("all_approve_dept"),
    F.first("actual_approve_node_num").alias("actual_approve_node_num"),
    F.first("approve_role").alias("approve_role"),
    F.first("job_level").alias("job_level"),
    F.first("avg_op_workday_hours").alias("avg_op_workday_hours"),
    F.first("actual_approve_person_times").alias("actual_approve_person_times"),
    F.first("avg_m1_workday_hours").alias("avg_m1_workday_hours"),
    F.first("avg_m2_workday_hours").alias("avg_m2_workday_hours"),
    F.first("avg_m3m4_workday_hours").alias("avg_m3m4_workday_hours"),
    F.first("avg_m5m6_workday_hours").alias("avg_m5m6_workday_hours"),
    F.first("avg_president_workday_hours").alias("avg_president_workday_hours"),
    F.first("avg_ceo_workday_hours").alias("avg_ceo_workday_hours"),
    F.first("avg_vp_cfo_workday_hours").alias("avg_vp_cfo_workday_hours"),
    # ===== 新增自然日平均时长字段 =====
    F.first("avg_op_natural_hours").alias("avg_op_natural_hours"),
    F.first("avg_m1_natural_hours").alias("avg_m1_natural_hours"),
    F.first("avg_m2_natural_hours").alias("avg_m2_natural_hours"),
    F.first("avg_m3m4_natural_hours").alias("avg_m3m4_natural_hours"),
    F.first("avg_m5m6_natural_hours").alias("avg_m5m6_natural_hours"),
    F.first("avg_president_natural_hours").alias("avg_president_natural_hours"),
    F.first("avg_ceo_natural_hours").alias("avg_ceo_natural_hours"),
    F.first("avg_vp_cfo_natural_hours").alias("avg_vp_cfo_natural_hours"),
    F.first("status").alias("status"),
    F.first("requestmark").alias("requestmark"),
    F.first("fixed_approval_node_count").alias("fixed_approval_node_count"),
    F.first("return_person_names").alias("return_person_names"),
    F.first("approve_person_names").alias("approve_person_names"),
    F.first("start_unix").alias("start_unix"),
    F.first("end_unix").alias("end_unix")
)

# ==================== 7. 最终字段映射 ====================
df_final = df_agg.select(
    F.coalesce(F.col("requestname"), F.lit("无流程信息")).alias("process_info"),
    F.col("StartApprovalTime").alias("start_approval_time"),
    F.col("EndApprovalTime").alias("end_approval_time"),
    F.col("process_start_time"),
    F.col("process_end_time"),
    F.coalesce(
        F.trim(F.regexp_replace(
            F.expr("substring(replace(WorkFlowName, '　', ' '), coalesce(instr(replace(WorkFlowName, '　', ' '), ' ')+1,1), length(WorkFlowName))"),
            r'`~`.*', ''
        )),
        F.lit("无流程类型")
    ).alias("process_type"),
    F.coalesce(F.col("DeptFullPath"), F.lit("")).alias("applicant_dept_full_path"),
    F.coalesce(
        F.when(F.instr(F.col("DeptFullPath"), "数字化服务中心")>0, F.lit("IT系统"))
        .when(F.instr(F.col("DeptFullPath"), "销售")>0, F.lit("销售系统"))
        .when(F.instr(F.col("DeptFullPath"), "财会")>0, F.lit("财务系统"))
        .when(F.instr(F.col("DeptFullPath"), "生产")>0, F.lit("生产系统"))
        .when(F.instr(F.col("DeptFullPath"), "农务")>0, F.lit("农务系统"))
        .when(F.instr(F.col("DeptFullPath"), "采购")>0, F.lit("采购系统"))
        .when(F.instr(F.col("DeptFullPath"), "审计")>0, F.lit("内审系统"))
        .when(F.instr(F.col("DeptFullPath"), "人力")>0, F.lit("人事系统"))
        .when(F.instr(F.col("DeptFullPath"), "公共关系")>0, F.lit("总工法系统"))
        .when(F.instr(F.col("DeptFullPath"), "质量")>0, F.lit("质量系统"))
        .otherwise(F.lit("其他")), F.lit("其他")
    ).alias("applicant_business_system"),
    F.col("business_approve_dept"),
    F.col("func_approve_dept"),
    F.coalesce(F.col("all_approve_dept"), F.lit("")).alias("all_approve_dept"),
    F.coalesce(F.col("actual_approve_node_num").cast(IntegerType()), F.lit(0)).alias("actual_approve_node_num"),
    F.col("approve_role"),
    F.coalesce(F.col("job_level"), F.lit("")).alias("job_level"),
    F.coalesce(F.col("avg_op_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_op_workday_hours"),
    F.coalesce(F.col("actual_approve_person_times").cast(IntegerType()), F.lit(0)).alias("actual_approve_person_times"),
    F.coalesce(F.col("avg_m1_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m1_workday_hours"),
    F.coalesce(F.col("avg_m2_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m2_workday_hours"),
    F.coalesce(F.col("avg_m3m4_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m3m4_workday_hours"),
    F.coalesce(F.col("avg_m5m6_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m5m6_workday_hours"),
    F.coalesce(F.col("avg_president_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_president_workday_hours"),
    F.coalesce(F.col("avg_ceo_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_ceo_workday_hours"),
    F.coalesce(F.col("avg_vp_cfo_workday_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_vp_cfo_workday_hours"),
    # ===== 新增自然日平均时长字段输出 =====
    F.coalesce(F.col("avg_op_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_op_natural_hours"),
    F.coalesce(F.col("avg_m1_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m1_natural_hours"),
    F.coalesce(F.col("avg_m2_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m2_natural_hours"),
    F.coalesce(F.col("avg_m3m4_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m3m4_natural_hours"),
    F.coalesce(F.col("avg_m5m6_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_m5m6_natural_hours"),
    F.coalesce(F.col("avg_president_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_president_natural_hours"),
    F.coalesce(F.col("avg_ceo_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_ceo_natural_hours"),
    F.coalesce(F.col("avg_vp_cfo_natural_hours").cast(DoubleType()), F.lit(0.0)).alias("avg_vp_cfo_natural_hours"),
    F.lit("办结").alias("process_status"),
    F.coalesce(
        F.when((F.col("end_unix") - F.col("start_unix")) > 0,
               F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2)
              ).otherwise(F.lit(0.0)),
        F.lit(0.0)
    ).cast(DoubleType()).alias("total_approval_natural_hours"),
    F.coalesce(
        F.when(F.col("total_work_seconds") > 0,
               F.round(F.col("total_work_seconds")/3600.0, 2)
              ).otherwise(F.lit(0.0)),
        F.lit(0.0)
    ).cast(DoubleType()).alias("total_approval_work_hours"),
    F.coalesce(F.col("IsReturn"), F.lit("否")).alias("is_return"),
    F.when(F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2) <= 24, F.lit("是")).otherwise(F.lit("否")).alias("is_fast_approval"),
    F.when((F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2) > 24) &
           (F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2) <= 120), F.lit("是")).otherwise(F.lit("否")).alias("is_normal_approval"),
    F.when(F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2) > 120, F.lit("是")).otherwise(F.lit("否")).alias("is_abnormal_approval"),
    F.when((F.round((F.col("end_unix") - F.col("start_unix"))/3600.0, 2) > 120) & (F.col("IsReturn") == F.lit("是")), F.lit("是")).otherwise(F.lit("否")).alias("is_low_efficiency_process"),
    F.col("requestid").cast(StringType()).alias("request_id"),
    F.coalesce(F.col("requestmark"), F.lit("")).alias("requestmark"),
    F.coalesce(F.col("fixed_approval_node_count").cast(IntegerType()), F.lit(0)).alias("fixed_approval_node_count"),
    F.coalesce(F.col("return_person_names"), F.lit("")).alias("return_person_names"),
    F.coalesce(F.col("approve_person_names"), F.lit("")).alias("approve_person_names")
).withColumn("dw_update_time", F.current_timestamp()) \
 .withColumn("source_flg", F.lit("OA"))

# ==================== 8. 写入 Hive ====================
# 去重：确保每个 request_id 唯一
df_final = df_final.dropDuplicates(["request_id"])
# 排除系统提醒工作流
df_final = df_final.filter(F.col("process_type") != "系统提醒工作流")

# 统计并打印唯一 request_id 数量
total_request_ids = df_final.count()
print(f"📊 流程效率结果表统计: 共写入 {total_request_ids} 条记录 (即 {total_request_ids} 个唯一 request_id)")

spark.sql(f"DROP TABLE IF EXISTS {target_table}")
df_final.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("encoding", "UTF-8") \
    .option("compression", "snappy") \
    .saveAsTable(target_table)

# 更新表注释
spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES ('comment' = 'OA流程效率分析结果表（筛选status=办结，剔除周末和节假日计算工作小时，使用硬编码2024节假日列表）')")
print(f"已写入表 {target_table}")
spark.stop()

/opt/tiger/spark/python/pyspark/context.py:238: FutureWarning: Python 3.6 support is deprecated in Spark 3.2.
  FutureWarning


📊 流程效率结果表统计: 共写入 234097 条记录 (即 234097 个唯一 request_id)
已写入表 dwd.dwd_oa_workflow_efficiency_result_f_1h
